## BicycleGAN

In [ ]:
import os

import torch
import torchvision.transforms as transforms
from datasets import load_dataset
from torchvision.transforms.functional import to_pil_image
from IPython.display import display
from PIL import Image, ImageDraw

from models import Generator
from utils.hf_utils import ImageDataset

In [ ]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def load_checkpoint(checkpoint_path, device="cuda"):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    latent_dim = checkpoint["latent_dim"]
    model = Generator(
		img_size=checkpoint["img_size"],
		in_channels=checkpoint["generator_in_channels"],
        latent_dim=latent_dim,
	)
    model = model.to(device)
    model.load_state_dict(checkpoint["G"])
    return model, latent_dim

### Edges2Shoes

In [ ]:
edges2shoes_checkpoint_path = os.path.join("checkpoints", "edges2shoes", "edges2shoes_25.pth")

In [ ]:
edges2shoes_model, edges2shoes_latent_dim = load_checkpoint(edges2shoes_checkpoint_path, device=device)

In [ ]:
hf_dataset = load_dataset("luethan2025/edges2shoes", split="val")
transform = transforms.Compose([
	transforms.ToTensor(),
	transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
edges_dataset = ImageDataset(hf_dataset, transform)

In [ ]:
random_idx = torch.randint(len(edges_dataset), (1,)).item()
edges_image, _ = edges_dataset[random_idx]
edges_image = edges_image.to(device)

edges_pil_image = to_pil_image(edges_image)
display(edges_pil_image)

In [ ]:
num_generated_shoes = 3
edge_batch = edges_image.unsqueeze(0).repeat(num_generated_shoes, 1, 1, 1)
shoes_latents = torch.randn(num_generated_shoes, edges2shoes_latent_dim, device=device)

edges2shoes_model.eval()
with torch.inference_mode():
    shoes_images = edges2shoes_model(edge_batch, shoes_latents)
    shoes_images = ((shoes_images + 1) / 2).clamp(0, 1).detach().cpu()

shoes_pil_images = [to_pil_image(image) for image in shoes_images]

generated_shoes = Image.new(
    "RGB",
    (num_generated_shoes * shoes_pil_images[0].width + (num_generated_shoes - 1),
     shoes_pil_images[0].height),
)
for index, shoes_pil_image in enumerate(shoes_pil_images):
    generated_shoes.paste(shoes_pil_image, (index * (shoes_pil_image.width), 0))

display(generated_shoes)

### Edges2Handbags

In [ ]:
edges2handbags_checkpoint_path = os.path.join("checkpoints", "edges2handbags", "edges2handbags_25.pth")

In [ ]:
edges2handbags_model, edges2handbags_latent_dim = load_checkpoint(edges2handbags_checkpoint_path, device=device)

In [ ]:
hf_dataset = load_dataset("luethan2025/edges2handbags", split="val")
transform = transforms.Compose([
	transforms.ToTensor(),
	transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
edges_dataset = ImageDataset(hf_dataset, transform)

In [ ]:
random_idx = torch.randint(len(edges_dataset), (1,)).item()
edges_image, _ = edges_dataset[random_idx]
edges_image = edges_image.to(device)

edges_pil_image = to_pil_image(edges_image)
display(edges_pil_image)

In [ ]:
num_generated_handbags = 3
edge_batch = edges_image.unsqueeze(0).repeat(num_generated_handbags, 1, 1, 1)
handbags_latents = torch.randn(num_generated_handbags, edges2handbags_latent_dim, device=device)

edges2handbags_model.eval()
with torch.inference_mode():
    handbags_images = edges2handbags_model(edge_batch, handbags_latents)
    handbags_images = ((handbags_images + 1) / 2).clamp(0, 1).detach().cpu()

handbags_pil_images = [to_pil_image(image) for image in handbags_images]

generated_handbags = Image.new(
    "RGB",
    (num_generated_handbags * handbags_pil_images[0].width + (num_generated_handbags - 1),
     handbags_pil_images[0].height),
)
for index, handbag_pil_image in enumerate(handbags_pil_images):
    generated_handbags.paste(handbag_pil_image, (index * (handbag_pil_image.width), 0))

display(generated_handbags)